<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача 1

*Загрузить готовый CLIP (авторы любезно предоставили обученные модели) здесь (https://huggingface.co/openai/models). Понять, как он работает: входные данные, выходные данные, параметры, что происходит внутри. Запустить CLIP на каком-нибудь простом примере, проанализировать результаты.*

In [ ]:
%matplotlib inline

In [ ]:
import transformers
import torch
import requests
import torchvision
from torchvision import transforms
from PIL import Image
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer
from collections import Counter
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn import metrics, preprocessing
import torch.nn.functional as F
import random
import numpy as np
import gc
from IPython.display import display
from torchvision.transforms.functional import to_tensor

In [ ]:
BATCH_SIZE = 128

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.eval()
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
# Retrieving text embeddings

inputs = tokenizer(["a photo of a cat", "a photo of a dog"], padding=True, return_tensors="pt")

with torch.inference_mode():
    text_features = model.get_text_features(**inputs)
    a = text_features.pooler_output[0]
    b = text_features.pooler_output[1]
    print(f'Similarity of labels: {torch.dot(a, b) / (torch.linalg.vector_norm(a) * torch.linalg.vector_norm(b))}')
    print(f'Embedding shape: {a.shape}')

In [ ]:
imagenette_data = torchvision.datasets.Imagenette('imagenette/', download=True)
data_loader = torch.utils.data.DataLoader(imagenette_data,
                                          batch_size=BATCH_SIZE,
                                          shuffle=False,
                                          num_workers=2,
                                          collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))

### Retrieving a sample

In [ ]:
classes_for_clip = ['A picture of ' + i[0] for i in imagenette_data.classes]

image, label = data_loader.dataset[random.randint(0, len(data_loader.dataset))]
print('Category labels according to Imagenette: ', end='')
print(*imagenette_data.classes[label], sep = ', ')
print()
print('Categories which clip would predict to:')
for option in classes_for_clip:
  print(f'    "{option}"')

print()
display(image)


### Prediction

In [ ]:
print("High-level model evaluation")

inputs = processor(text=classes_for_clip, images=image, padding=True, return_tensors="pt")
outputs = model(**inputs)

print(f'Logits: {outputs.logits_per_image[0].detach()}')
print(f'Logits SOFTMAX: {outputs.logits_per_image[0].softmax(dim=0).detach()}')

#### Comparing logits
Here we compare logits retrieved via high-level transformers API and our lower-level realisation.

In [ ]:
with torch.inference_mode():
  txt_inputs = tokenizer(text=classes_for_clip, padding=True, return_tensors="pt")

  text_features = model.get_text_features(**txt_inputs)
  text_emb = text_features.pooler_output.detach()

  img_inputs = processor(images=image, padding=True, return_tensors='pt')
  image_emb = model.get_image_features(**img_inputs).pooler_output.detach()

  text_emb = torch.div(text_emb, torch.norm(text_emb, 2, dim=1).repeat(512, 1).T)
  image_emb /= torch.norm(image_emb, 2, dim=1)

  print(f'Feature shapes\n    Text: {text_emb.shape}\n    Image: {image_emb.shape}')

As we can see, pooler_output gives us already projected embeddings, spesifically, 768-dimensional embedding of image transformed to 512-dimensional one

Looks like text embeddings were 512-dimensional originally.

In [ ]:
  predictions = (image_emb @ text_emb.T) * (np.exp(model.logit_scale.item()))
  print(f'Logits: {predictions}')
  predictions = predictions.softmax(dim=1)
  print(f'Logits SOFTMAX: {predictions}')


Well, they are same to the high-level realisation

#### Prediction from CLIP

In [ ]:
  display(image)
  print('\nSo clip said it is:')
  for i, option in enumerate(classes_for_clip):
    print(f'    {i + 1}. {option} {(predictions[0][i]) * 100}%')
